# VQA 최적화 솔루션 — InternVL2-8B (Google Colab)

| 항목 | 사양 |
|------|------|
| 모델 | `OpenGVLab/InternVL2-8B` (~8B) |
| 이미지 크기 | **448×448** (InternVL 기본 권장 해상도) |
| 이미지 전처리 | ImageNet 정규화 + **학습 시 증강** |
| 증강 | Flip / ColorJitter / Rotation / RandomAffine |
| 학습 | LoRA r=16, 3 Epochs, Cosine LR, Gradient Clipping |
| 레이블 마스킹 | 답변 토큰만 loss 계산 |
| 추론 | **Logit 직접 비교** (a/b/c/d 확률) |
| 검증 지표 | **val_accuracy** 기준 best 저장 |
| 학습 샘플 수 | `NUM_TRAIN_SAMPLES = 200` (에폭당 약 1시간, A100 기준) |

> **드라이브 구조 가정:**
> ```
> MyDrive/
>   subset_train.csv
>   subset_val.csv
>   test.csv
>   train/   ← 학습 이미지
>   dev/     ← 검증 이미지
>   test/    ← 테스트 이미지
> ```

## 0. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 라이브러리 설치

> Colab에는 PyTorch/torchvision이 이미 설치되어 있으므로 재설치하지 않습니다.

In [ ]:
import os

ZIP_PATH    = "/content/drive/My Drive/test.zip"
EXTRACT_DIR = "/content/"

# 이미 압축 해제된 경우 스킵
if not os.path.exists("/content/train"):
    print("압축 해제 중...")
    !unzip -q "{ZIP_PATH}" -d "{EXTRACT_DIR}"
    print("완료!")
else:
    print("이미 압축 해제되어 있습니다. 스킵합니다.")

이미 압축 해제되어 있습니다. 스킵합니다.


In [ ]:
# Colab 전용 설치 (torch/torchvision은 이미 설치되어 있음)
!pip install -q \
    "transformers>=4.43.2,<5.0.0" \
    "accelerate>=0.34.2" \
    "peft>=0.13.2" \
    "bitsandbytes>=0.43.3" \
    einops timm sentencepiece tiktoken

## 2. CUDA 환경 확인

In [ ]:
import torch
print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("bfloat16 지원:", torch.cuda.is_bf16_supported())

PyTorch 버전: 2.10.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
bfloat16 지원: True


## 3. 라이브러리 임포트 & 하이퍼파라미터

In [ ]:
import os, math, random
from dataclasses import dataclass
from typing import Any, List

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# GPU에 따라 compute dtype 자동 선택 (A100/L4 → bfloat16, T4 → float16)
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("Compute dtype:", COMPUTE_DTYPE)

# ── 하이퍼파라미터 ──────────────────────────────────────────────────────────────
MODEL_ID       = "OpenGVLab/InternVL2-8B"
IMAGE_SIZE     = 448      # InternVL 권장 해상도 (28×28 패치 기준)
NUM_EPOCHS     = 3
BATCH_SIZE     = 1
GRAD_ACCUM     = 4        # 유효 배치 = 4
LR             = 1e-4
MAX_GRAD_NORM  = 1.0
LORA_R         = 16
LORA_ALPHA     = 32
WARMUP_RATIO   = 0.05
SEED           = 42

# ── 훈련 샘플 수 제한 (에폭당 약 1시간 목표) ────────────────────────────────────
# A100: ~200개 ≈ 1시간 / L4: ~100개 ≈ 1시간 / T4: ~60개 ≈ 1시간
NUM_TRAIN_SAMPLES = 2300

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
DRIVE_BASE    = "/content/drive/My Drive"
DATA_PATH     = DRIVE_BASE
WORKSPACE_DIR = f"{DRIVE_BASE}/content"
SAVE_DIR      = f"{WORKSPACE_DIR}/internvl2_vqa_lora"
os.makedirs(SAVE_DIR, exist_ok=True)

# ImageNet 정규화 (InternVL2 표준)
IMAGENET_MEAN  = (0.485, 0.456, 0.406)
IMAGENET_STD   = (0.229, 0.224, 0.225)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Device: cuda
Compute dtype: torch.bfloat16


## 4. 데이터 로드

In [ ]:
def fix_image_path(p: str) -> str:
    """CSV의 상대 경로를 Drive 절대 경로로 변환 (예: train/img.jpg → /content/drive/My Drive/train/img.jpg)"""
    p = str(p).replace('\\', '/')
    if p.startswith('/'):
        return p  # 이미 절대 경로면 그대로
    return f"{DRIVE_BASE}/{p}"


train_df = pd.read_csv(f"{DATA_PATH}/subset_train.csv")
val_df   = pd.read_csv(f"{DATA_PATH}/subset_val.csv")
test_df  = pd.read_csv(f"{DATA_PATH}/test.csv")

# 이미지 경로 → Drive 절대 경로로 변환
train_df['path'] = train_df['path'].apply(fix_image_path)
val_df['path']   = val_df['path'].apply(fix_image_path)
test_df['path']  = test_df['path'].apply(fix_image_path)

# ── 훈련 샘플 수 제한 (에폭당 ~1시간 목표) ────────────────────────────────────
if NUM_TRAIN_SAMPLES < len(train_df):
    train_df = train_df.sample(n=NUM_TRAIN_SAMPLES, random_state=SEED).reset_index(drop=True)

print(f"Train: {len(train_df)}개 / Val: {len(val_df)}개 / Test: {len(test_df)}개")
print("\n답변 분포 (train):")
print(train_df["answer"].value_counts().sort_index())

Train: 2300개 / Val: 258개 / Test: 5074개

답변 분포 (train):
answer
a    580
b    577
c    543
d    600
Name: count, dtype: int64


## 5. 이미지 전처리 & 증강

In [ ]:
def get_train_transform(image_size: int = IMAGE_SIZE):
    """학습용: 증강 + 정규화 (PIL → Tensor)"""
    return transforms.Compose([
        transforms.Resize((image_size, image_size),
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(
            brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05
        ),
        transforms.RandomRotation(degrees=15, fill=128),
        transforms.RandomAffine(
            degrees=0, translate=(0.05, 0.05), fill=128
        ),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_val_transform(image_size: int = IMAGE_SIZE):
    """검증/추론용: 리사이즈 + 정규화만"""
    return transforms.Compose([
        transforms.Resize((image_size, image_size),
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


train_transform = get_train_transform()
val_transform   = get_val_transform()
print("학습 transform:", train_transform)

학습 transform: Compose(
    Resize(size=(448, 448), interpolation=bicubic, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.7, 1.3), contrast=(0.7, 1.3), saturation=(0.8, 1.2), hue=(-0.05, 0.05))
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=128)
    RandomAffine(degrees=[0.0, 0.0], translate=(0.05, 0.05), fill=128)
    ToTensor()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
)


## 6. 모델 & Tokenizer 로드

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
)

base_model = AutoModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map={"": 0},
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)

num_image_token = base_model.num_image_token
img_context_token_id = tokenizer.convert_tokens_to_ids("<IMG_CONTEXT>")
print(f"이미지 토큰 수: {num_image_token}")
print(f"<IMG_CONTEXT> token_id: {img_context_token_id}")
base_model.img_context_token_id = img_context_token_id

# InternVL2-8B는 InternLM2 백본 사용 → wqkv/wo/w1/w2/w3 구조
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=["wqkv", "wo", "w1", "w2", "w3"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

InternLM2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

이미지 토큰 수: 256
<IMG_CONTEXT> token_id: 92546
trainable params: 37,748,736 || all params: 8,113,114,112 || trainable%: 0.4653


## 7. 프롬프트 & 대화 포맷

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a visual question answering expert specializing in recyclable materials. "
    "Analyze the image carefully and answer with exactly one letter: a, b, c, or d. "
    "No explanation — just a single lowercase letter."
)

IMG_START = "<img>"
IMG_END   = "</img>"
IMG_CTX   = "<IMG_CONTEXT>"


def build_mc_prompt(question: str, a: str, b: str, c: str, d: str) -> str:
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요."
    )


def build_conversation_text(
    question: str, a: str, b: str, c: str, d: str,
    answer: str = None
) -> str:
    """InternVL2 chat template 형식으로 대화 텍스트 생성"""
    image_tokens = IMG_START + IMG_CTX * num_image_token + IMG_END
    user_content = image_tokens + "\n" + build_mc_prompt(question, a, b, c, d)

    conversation = [
        {"role": "system", "content": SYSTEM_INSTRUCT},
        {"role": "user",   "content": user_content},
    ]
    if answer is not None:
        conversation.append({"role": "assistant", "content": answer})

    return tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=(answer is None),
    )

## 8. Dataset & DataCollator

In [ ]:
CHOICES = ["a", "b", "c", "d"]


class VQAMCDataset(Dataset):
    def __init__(self, df, train: bool = True):
        self.df        = df.reset_index(drop=True)
        self.train     = train
        self.transform = train_transform if train else val_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row          = self.df.iloc[i]
        img          = Image.open(row["path"]).convert("RGB")
        pixel_values = self.transform(img)
        answer       = str(row["answer"]).strip().lower() if self.train else None
        text         = build_conversation_text(
            str(row["question"]), str(row["a"]), str(row["b"]),
            str(row["c"]),        str(row["d"]), answer=answer
        )
        return {"text": text, "pixel_values": pixel_values, "answer": answer}


@dataclass
class DataCollatorV2:
    tokenizer: Any
    train: bool = True

    def __call__(self, batch: List[dict]):
        texts        = [s["text"] for s in batch]
        pixel_values = torch.stack([s["pixel_values"] for s in batch])

        enc = self.tokenizer(
            texts, padding=True, truncation=True,
            max_length=2048, return_tensors="pt",
        )

        result = {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "pixel_values":   pixel_values,
            "image_flags":    torch.ones(len(batch), 1, dtype=torch.long),
        }

        if self.train:
            labels = enc["input_ids"].clone()

            for i, sample in enumerate(batch):
                answer   = sample["answer"]
                full_ids = enc["input_ids"][i]
                real_len = enc["attention_mask"][i].sum().item()

                # answer 토큰 ID 찾기
                ans_id = self.tokenizer.encode(
                    answer, add_special_tokens=False
                )[0]

                # 마지막 위치부터 역방향으로 answer 토큰 위치 탐색
                ans_pos = -1
                for pos in range(real_len - 1, -1, -1):
                    if full_ids[pos].item() == ans_id:
                        ans_pos = pos
                        break

                # 답변 토큰만 남기고 나머지 마스킹
                labels[i, :] = -100
                if ans_pos >= 0:
                    labels[i, ans_pos] = full_ids[ans_pos]

            result["labels"] = labels

        return result

## 9. DataLoader

In [ ]:
train_ds = VQAMCDataset(train_df, train=True)
val_ds   = VQAMCDataset(val_df,   train=False)

collator_train = DataCollatorV2(tokenizer, train=True)
collator_val   = DataCollatorV2(tokenizer, train=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collator_train, num_workers=0, pin_memory=False
)
print(f"Train 배치 수: {len(train_loader)} / Val: {len(val_df)}개")

Train 배치 수: 2300 / Val: 258개


## 10. Logit 추론 함수

In [ ]:
# a, b, c, d 토큰 ID
choice_token_ids = []
for ch in CHOICES:
    ids = tokenizer.encode(ch, add_special_tokens=False)
    choice_token_ids.append(ids[0])
    print(f"'{ch}' → token_id = {ids[0]}")
choice_ids_tensor = torch.tensor(choice_token_ids, device=device)


@torch.no_grad()
def predict_logit(img: Image.Image, question: str,
                  a: str, b: str, c: str, d: str) -> str:
    """이미지 + 질문을 받아 a/b/c/d 로짓 비교로 정답 반환"""
    pv   = val_transform(img).unsqueeze(0).to(device)
    text = build_conversation_text(question, a, b, c, d, answer=None)
    enc  = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=2048
    ).to(device)

    image_flags = torch.ones(1, 1, dtype=torch.long).to(device)
    with torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE):
        out = model(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            pixel_values=pv,
            image_flags=image_flags,
        )

    last_logits = out.logits[0, -1, choice_ids_tensor]
    return CHOICES[last_logits.argmax().item()]


def evaluate_accuracy(df: pd.DataFrame, desc: str = "Val") -> float:
    correct = 0
    model.eval()
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc, leave=False):
        img  = Image.open(row["path"]).convert("RGB")
        pred = predict_logit(
            img, str(row["question"]),
            str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        )
        if pred == str(row["answer"]).strip().lower():
            correct += 1
    return correct / len(df)

'a' → token_id = 264
'b' → token_id = 260
'c' → token_id = 271
'd' → token_id = 273


## 11. Fine-tuning

In [ ]:
num_update_steps = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps = int(num_update_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_update_steps
)
# bfloat16은 GradScaler가 불필요하지만, float16 폴백 시를 위해 enabled 조건 적용
scaler = torch.amp.GradScaler("cuda", enabled=(COMPUTE_DTYPE == torch.float16))

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    # ── 학습 ──
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE):
            out  = model(**batch)
            loss = out.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            avg = running_loss / GRAD_ACCUM
            pbar.set_postfix({"loss": f"{avg:.4f}",
                              "lr":   f"{scheduler.get_last_lr()[0]:.2e}"})
            running_loss = 0.0

    # ── 검증: Accuracy ──
    val_acc = evaluate_accuracy(val_df, desc=f"Epoch {epoch+1} [val acc]")
    print(f"[Epoch {epoch+1}] val_accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"  ✓ Best model saved → {SAVE_DIR} (val_acc={val_acc*100:.2f}%)")

print(f"\n학습 완료. Best val_accuracy: {best_val_acc*100:.2f}%")

Epoch 1/3 [train]:   0%|          | 0/2300 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Epoch 1 [val acc]:   0%|          | 0/258 [00:00<?, ?it/s]

[Epoch 1] val_accuracy: 0.6550 (65.50%)
  ✓ Best model saved → /content/drive/My Drive/content/internvl2_vqa_lora (val_acc=65.50%)


Epoch 2/3 [train]:   0%|          | 0/2300 [00:00<?, ?it/s]

Epoch 2 [val acc]:   0%|          | 0/258 [00:00<?, ?it/s]

[Epoch 2] val_accuracy: 0.6744 (67.44%)
  ✓ Best model saved → /content/drive/My Drive/content/internvl2_vqa_lora (val_acc=67.44%)


Epoch 3/3 [train]:   0%|          | 0/2300 [00:00<?, ?it/s]

Epoch 3 [val acc]:   0%|          | 0/258 [00:00<?, ?it/s]

[Epoch 3] val_accuracy: 0.6938 (69.38%)
  ✓ Best model saved → /content/drive/My Drive/content/internvl2_vqa_lora (val_acc=69.38%)

학습 완료. Best val_accuracy: 69.38%


## 12. 테스트 추론 & 제출

In [ ]:
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Test Inference"):
    row  = test_df.iloc[i]
    img  = Image.open(row["path"]).convert("RGB")
    pred = predict_logit(
        img, str(row["question"]),
        str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
    )
    preds.append(pred)

print("\n예측 분포:")
print(pd.Series(preds).value_counts().sort_index())

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv(f"{WORKSPACE_DIR}/submission.csv", index=False)
print(f"Saved: {WORKSPACE_DIR}/submission.csv")
submission.head(10)

Test Inference:   0%|          | 0/5074 [00:00<?, ?it/s]


예측 분포:
a    1283
b    1245
c    1278
d    1268
Name: count, dtype: int64
Saved: /content/drive/My Drive/content/submission.csv


,id,answer
0,test_0001.jpg,d
1,test_0002.jpg,d
2,test_0003.jpg,c
3,test_0004.jpg,a
4,test_0005.jpg,b
5,test_0006.jpg,c
6,test_0007.jpg,a
7,test_0008.jpg,b
8,test_0009.jpg,b
9,test_0010.jpg,c
